# 06. Memory Optimization & Type Precision: Beginner Guide

### 🌟 What is Memory Precision & Type Downcasting in NumPy?
By default, NumPy uses 64-bit numbers (`float64`, `int64`). By carefully choosing smaller precision types (`float32`, `int16`, `int8`) when appropriate, you can cut RAM usage by 50% to 75% without sacrificing calculation accuracy.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Typecasting (`.astype`)**: Converting dtypes across int8, int16, int32, int64, float32, and float64.
- **Precision Management**: Downcasting float64 to float32 cutting memory footprint by exactly 50%.
- **Integer Overflow Bounds**: Understanding two's complement boundary rollover in fixed-width integers.
- **Inspecting Numeric Limits**: Covers `np.iinfo()` and `np.finfo()`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### 🔹 Typecasting with `.astype()`
Casts transaction amounts from float64 (8 bytes) to float32 (4 bytes). Checking and validating data types prevents subtle runtime errors and ensures subsequent mathematical or string operations behave properly. **Tip:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.

**Syntax:** `amounts.astype(np.float32)`


In [2]:
amounts_f32 = amounts.astype(np.float32)
print('Original Dtype:', amounts.dtype, '-> New Dtype:', amounts_f32.dtype)

Original Dtype: float64 -> New Dtype: float32


### 🔹 Precision Management: 50% RAM Reduction
Measures memory savings from downcasting clean transaction amounts. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `amounts.nbytes`


In [3]:
print(f'float64 Memory: {amounts.nbytes / 1024:.2f} KB')
print(f'float32 Memory: {amounts_f32.nbytes / 1024:.2f} KB (exactly 50% RAM saved)')

float64 Memory: 111.42 KB
float32 Memory: 55.71 KB (exactly 50% RAM saved)


### 🔹 Integer Overflow Boundary Testing
Demonstrates rollover when casting high account ages into `int8`. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.array([127], dtype=np.int8) + 1`


In [4]:
overflow_demo = np.array([125, 126, 127], dtype=np.int8)
overflow_demo += 1
print('int8 Two\'s Complement Overflow Result (127 + 1 -> -128):', overflow_demo)

int8 Two's Complement Overflow Result (127 + 1 -> -128): [ 126  127 -128]


### 🔹 Inspecting Numeric Limits with `iinfo` & `finfo`
Inspects machine limits for `int8`, `int16`, and `float32`. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.iinfo(np.int8)`


In [5]:
print('int8 Range:', np.iinfo(np.int8).min, 'to', np.iinfo(np.int8).max)
print('int16 Range:', np.iinfo(np.int16).min, 'to', np.iinfo(np.int16).max)
print('float32 Machine Epsilon:', np.finfo(np.float32).eps)

int8 Range: -128 to 127
int16 Range: -32768 to 32767
float32 Machine Epsilon: 1.1920929e-07


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Automated Safe Downcasting on Transaction Vectors

**Approach:** Write a function to downcast transaction amounts and account ages to the smallest safe dtype without overflow.
**Syntax:** `np.iinfo` checks


In [6]:
def safe_downcast(arr):
    if np.issubdtype(arr.dtype, np.floating):
        return arr.astype(np.float32)
    elif np.issubdtype(arr.dtype, np.integer):
        for dt in [np.int8, np.int16, np.int32]:
            if arr.min() >= np.iinfo(dt).min and arr.max() <= np.iinfo(dt).max:
                return arr.astype(dt)
    return arr

print('Downcasted Fraud Flags Dtype:', safe_downcast(fraud_flags).dtype)
print('Downcasted Amounts Dtype:', safe_downcast(amounts).dtype)

Downcasted Fraud Flags Dtype: int8
Downcasted Amounts Dtype: float32
